In [0]:
/*
Static lookup tables to represent fiscal quarters.
*/
with dates as (
  select cast(m as date) as m, fq, fy, q
  , last_day(m) as last_day_of_month, year(m) as cy, month(m) as cm
  from (
    values 
      /*('2025-02-01', 'FY26-Q1', 2026, 1),
      ('2025-03-01', 'FY26-Q1', 2026, 1),
      ('2025-04-01', 'FY26-Q1', 2026, 1),
      ('2025-05-01', 'FY26-Q2', 2026, 2),
      ('2025-06-01', 'FY26-Q2', 2026, 2),
      ('2025-07-01', 'FY26-Q2', 2026, 2),
      ('2025-08-01', 'FY26-Q3', 2026, 3),
      ('2025-09-01', 'FY26-Q3', 2026, 3),
      ('2025-10-01', 'FY26-Q3', 2026, 3),
      ('2025-11-01', 'FY26-Q4', 2026, 4),
      ('2025-12-01', 'FY26-Q4', 2026, 4),
      ('2026-01-01', 'FY26-Q4', 2026, 4),*/
      ('2026-02-01', 'FY27-Q1', 2027, 1),
      ('2026-03-01', 'FY27-Q1', 2027, 1),
      ('2026-04-01', 'FY27-Q1', 2027, 1),
      ('2026-05-01', 'FY27-Q2', 2027, 2),
      ('2026-06-01', 'FY27-Q2', 2027, 2),
      ('2026-07-01', 'FY27-Q2', 2027, 2),
      ('2026-08-01', 'FY27-Q3', 2027, 3),
      ('2026-09-01', 'FY27-Q3', 2027, 3),
      ('2026-10-01', 'FY27-Q3', 2027, 3),
      ('2026-11-01', 'FY27-Q4', 2027, 4),
      ('2026-12-01', 'FY27-Q4', 2027, 4),
      ('2027-01-01', 'FY27-Q4', 2027, 4) 
  ) as dates(m, fq, fy, q)
),

-- Resolve :ae_email to a list of AE emails.
-- If :ae_email is an AE, returns just that email. If a manager, returns all AEs reporting to them.
ae_list as (
  select Email as ae_email, user_name, IsAE, level as sales_level
  from main.gtm_silver.individual_hierarchy_salesforce
  where snapshot_date = (select max(snapshot_date) from main.gtm_silver.individual_hierarchy_salesforce)
  --and IsAE = true
  and IsActive = true
  and Business_Unit = :business_unit
  and Region_Level_1 = :region_level_1
  and Region_Level_2 = :region_level_2
  and concatenated_emails like '%' || :ae_email || '%'
),

financial_quarters as (
  select fq, fy, q
    , max(last_day_of_month) as fiscal_quarter_end_date
    , min(m) as fiscal_quarter_start_date
    , case when getdate() > fiscal_quarter_end_date then q else null end as last_closed_q
    , case when getdate() > fiscal_quarter_end_date then true else false end as is_quarter_closed
    , sum(day(last_day_of_month)) as days_in_quarter
    , case when getdate() >= fiscal_quarter_start_date and current_date() <= fiscal_quarter_end_date then true else false end as is_current_fiscal_quarter
    , case when current_date() >= make_date(fy - 1, 2, 1) and current_date() <= make_date(fy, 1, 31) then 1 else 0 end as is_current_fiscal_year
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily)  as latest_usage_date
    , greatest(0, least(days_in_quarter, datediff(fiscal_quarter_end_date, latest_usage_date))) as days_left_in_quarter
    , case when is_current_fiscal_quarter then q else 0 end as current_quarter_number
  from dates
  group by fq, fy, q
),

-- Account-level targets joined to account_dim for deployable_account_name
-- Region filtering applied via account_dim (targets_account uses broader region granularity)
targets as (
  SELECT ae.ae_email, ad.deployable_account_name,
    ta.fiscal_year, ta.fiscal_quarter_number as fiscal_quarter,
    SUM(ta.dbu_dollar_target) as fin_target
  FROM main.gtm_silver.targets_account ta
  INNER JOIN main.gtm_silver.account_dim ad ON ta.account_id = ad.account_id
  INNER JOIN ae_list ae ON ad.concatenated_emails LIKE '%' || ae.ae_email || '%'
  WHERE ad.business_unit = :business_unit
  AND ad.region_level_1 = :region_level_1
  AND ad.region_level_2 = :region_level_2
  GROUP BY ae.ae_email, ad.deployable_account_name, ta.fiscal_year, ta.fiscal_quarter_number
),

-- Individual-level targets for the 'All' rollup row (uses each AE's own target, not sum of account targets)
targets_individual_ae as (
  SELECT ae.ae_email,
    CAST(ti.fiscal_year AS INT) as fy,
    CAST(ti.fiscal_quarter AS INT) as q,
    SUM(ti.dollars) as fin_target
  FROM main.gtm_silver.targets_individual ti
  INNER JOIN ae_list ae ON ti.Email = ae.ae_email
  WHERE ti.type_target = 'dbu'
  GROUP BY ae.ae_email, ti.fiscal_year, ti.fiscal_quarter
),

-- Individual-level forecast for the 'All' rollup row (uses each AE's own forecasts, not sum of account forecasts)
forecast_individual_ae as (
  SELECT ae.ae_email,
    d.fy,
    d.q,
    fi.submitted_my_call,
    fi.submitted_direct_field_consumption_forecast
  FROM main.gtm_silver.forecast_consumption_mcp_individual fi
  INNER JOIN ae_list ae ON fi.Email = ae.ae_email
  INNER JOIN financial_quarters d ON d.fiscal_quarter_end_date = fi.fiscal_quarter_end_date
  WHERE fi.snapshot_date = (SELECT MAX(snapshot_date) FROM main.gtm_silver.forecast_consumption_mcp_individual)
),

-- Account-level sales forecast aggregated to deployable_account_name + quarter
sales_forecast as (
  select ae.ae_email, ad.deployable_account_name, fa.forecast_fiscal_quarter_end_date as fiscal_quarter_end_date
    , sum(fa.submitted_ae_forecast) as submitted_my_call
    , sum(fa.submitted_weighted_projection) as submitted_weighted_projection
    , sum(coalesce(fa.submitted_organic_growth_baseline_in_plan, 0) + coalesce(fa.submitted_ds_forecast_out_of_plan, 0) + coalesce(fa.submitted_in_plan_adjustment, 0) + coalesce(fa.submitted_open_in_plan_usecases, 0)) as submitted_direct_field_consumption_forecast
    , sum(fa.current_ds_forecast) as current_ds_forecast -- per-account DS model forecast (sums correctly across deployable accounts)
  from main.gtm_silver.forecast_consumption_mcp_account as fa
  inner join main.gtm_silver.account_dim ad on fa.account_id = ad.account_id
  inner join ae_list ae on ad.concatenated_emails like '%' || ae.ae_email || '%'
  where fa.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_account)
  group by ae.ae_email, ad.deployable_account_name, fa.forecast_fiscal_quarter_end_date
),

actuals as (
  select ae.ae_email, c.deployable_account_name, c.fiscal_quarter_start_date, sum(c.dbu_dollars_qtd) as dbu_actuals
  , sum(c.dbu_dollars_t7d_avg) as dbu_dollars_t7d_avg, sum(c.dbu_dollars_t28d_avg) as dbu_dollars_t28d_avg
  , sum(c.dbu_dollars_t7d_avg_prev) as dbu_dollars_t7d_avg_prev, sum(c.dbu_dollars_t28d_avg_prev) as dbu_dollars_t28d_avg_prev
  from ae_list ae
  inner join main.gtm_gold.materialized__view_account_obt as c
    on c.concatenated_emails like '%' || ae.ae_email || '%'
  left outer join main.gtm_silver.account_dim as b
    on c.account_id = b.account_id
  where b.business_unit = :business_unit
  and b.region_level_1 = :region_level_1
  and b.region_level_2 = :region_level_2
  group by ae.ae_email, c.deployable_account_name, c.fiscal_quarter_start_date
),

-- Deployable account-level target/forecast/actuals (no proportional allocation needed)
target_forecast_actuals as (
  select d.fy, d.q, a.ae_email, a.deployable_account_name, a.fiscal_quarter_start_date, a.dbu_actuals
    , a.dbu_dollars_t7d_avg, a.dbu_dollars_t28d_avg, a.dbu_dollars_t7d_avg_prev, a.dbu_dollars_t28d_avg_prev
    , t.fin_target
    , f.submitted_my_call
    , f.submitted_direct_field_consumption_forecast
    , f.current_ds_forecast
    , f.submitted_weighted_projection
    , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
    , case when d.is_quarter_closed then a.dbu_actuals else f.submitted_my_call end dbu_actuals_or_forecast
    , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email, a.deployable_account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
    , coalesce(a.dbu_dollars_t7d_avg, dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj --for future quarters, use the latest t7d available.
    , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email, a.deployable_account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
    , coalesce(a.dbu_dollars_t28d_avg, dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj --for future quarters, use the latest t28d available.
    , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email, a.deployable_account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
    , coalesce(a.dbu_dollars_t7d_avg_prev, dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj --for future quarters, use the latest t7d_prev available.
    , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email, a.deployable_account_name order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
    , coalesce(a.dbu_dollars_t28d_avg_prev, dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj --for future quarters, use the latest t28d_prev available.
    , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
    , try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter) as target_t7d  

  from actuals as a
  inner join financial_quarters d
  on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
  left outer join sales_forecast as f 
  on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.ae_email = a.ae_email and f.deployable_account_name = a.deployable_account_name
  left outer join targets as t 
  on t.fiscal_year = d.fy and t.fiscal_quarter = d.q and t.ae_email = a.ae_email and t.deployable_account_name = a.deployable_account_name
),

usecases_filtered as (
  select ae.ae_email, use_case_detail.deployable_account_name, usecase_id, usecase_name, estimated_monthly_dollar_dbus, target_onboarding_date, target_live_date
    , dateadd(day, 14, date_trunc('month',target_onboarding_date)) as target_onboarding_date_15 
    , dateadd(day, 14, date_trunc('month',target_live_date)) as target_live_date_15
    , datediff(target_live_date, target_onboarding_date) as total_ramping_days 
    , date_format(dateadd(year, +1, dateadd(month, -1, target_onboarding_date)), "'FY'yy'-Q'Q") as target_onboarding_date_fq
    , date_format(dateadd(year, +1, dateadd(month, -1, target_live_date)), "'FY'yy'-Q'Q") as target_live_date_fq
    , concat('<a href="https://databricks.lightning.force.com/lightning/r/UseCase__c/', usecase_id, '/view" targe="_blank">', usecase_name, '</a>') as usecase_url
    , coalesce(num_of_blockers, 0) as num_of_blockers
    , case
        when days_in_stage <= 30 or days_in_stage is null then '0-30 days'
        when days_in_stage > 30 and days_in_stage <= 60 then '31-60 days'
        when days_in_stage > 60 and days_in_stage <= 120 then '61-120 days'
        when days_in_stage > 120 then '120+ days'
      end as days_in_stage_bucket
    , date_diff(DAY, current_date(), last_day(target_live_date)) as days_to_go_live
    , case when days_to_go_live < 0 then true else false end go_live_in_the_past
    , date_diff(DAY, current_date(), last_day(target_onboarding_date)) as days_to_onboarding

     --Check hygiene issues and risks
    ,case
      -- Hygiene Issues
      when go_live_in_the_past then named_struct('category', 'Hygiene', 'msg', 'Go live date in the past')
      when implementation_status is null then named_struct('category', 'Hygiene', 'msg', 'Health status not defined')
      when days_to_onboarding < 0 and stage_number < 5 then named_struct('category', 'Hygiene', 'msg', 'Past Onboarding date / not U5') 
      -- Risks
      when days_to_go_live < 30 and stage_number < 5 then named_struct('category', 'Warning', 'msg', 'Go live < 30 / Not U5')
      when days_to_go_live < 30 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Red')
      when days_to_go_live < 30 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Yellow')
      when days_to_go_live < 30 then named_struct('category', 'Warning', 'msg', 'Go live < 30')
      when days_to_onboarding between 0 and 30 and stage_number <5 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Red')
      when days_to_onboarding between 0 and 30 and stage_number <5 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Yellow')
      when days_to_onboarding between 0 and 30 and stage_number <=3 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / <=U3')
      when days_to_onboarding between 0 and 30 and stage_number =4 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / U4')      
      -- opportunities to accelerate
      when days_to_go_live < 60 and stage_number = 5 then named_struct('category', 'Opportunity', 'msg', 'Go live < 60 / U5')
      when days_to_onboarding between 0 and 60 and stage_number between 2 and 3 then named_struct('category', 'Opportunity', 'msg', 'Tech win to accelerate')
    end as uco_info
  , coalesce(implementation_status, 'Unknown') as implementation_status
    
  from gtm_silver.use_case_detail
  inner join ae_list ae on use_case_detail.concatenated_emails like '%' || ae.ae_email || '%'
  where Business_Unit = :business_unit
  and sales_subregion_level_1 = :region_level_1
  and sales_subregion_level_2 = :region_level_2
  and is_incremental = true --Excludes upgrades
  and stage_number <= 5 --Filter out 'Disqualified', 'Lost' and 'Live' UCOs.
  and coalesce(estimated_monthly_dollar_dbus, 0) > 0 -- Excludes zero-valued use cases.
  --and usecase_id = 'aAv8Y000000CsLuSAK' 
  /* test cases 
  aAv8Y000000CsLuSAK (Feb25->Sep25), aAvVp000000Uc6IKAS (May25->Sep25), aAv8Y000000lD0ySAE (Jun25->Dec25)
  aAvVp000000W9JVKA0 (Apr25->May25), aAvVp000000d2YEKAY (Feb25->Jul25), aAvVp000000WAqfKAG (Jun25->Jan26)
  */
),

incremental_projections as (
  select uco.ae_email, uco.deployable_account_name, uco.usecase_id, d.fq, d.fy, d.q, d.m, d.cm, d.last_day_of_month
    , uco.target_onboarding_date, uco.target_onboarding_date_fq, uco.target_live_date, uco.target_live_date_fq, uco.target_onboarding_date_15, uco.target_live_date_15
    , uco.total_ramping_days, uco.estimated_monthly_dollar_dbus, uco.implementation_status, uco.usecase_url, uco.num_of_blockers 
    , datediff(getdate(), target_onboarding_date_15) as current_ramping_days 
    -- Calculate this month's baseline for each use case, .i.e. how much are they consuming today? This is used to calculate the actual incremental consumption at the next step.
    -- We assume that the onboarding date and live date occur on day 15 of the month.
    ,case when d.m between uco.target_onboarding_date and uco.target_live_date then 1 else 0 end as is_onboarding
  
    ,case         
        -- if UCO not onboarded yet (i.e. the onboarding date is in the future), then no dbus are generated for the current month.
        when target_onboarding_date_15 > getdate() then 0
        -- if UCO is already live, then it should already realise the expected monthly $DBUs.
        when getdate() > target_live_date_15 then estimated_monthly_dollar_dbus
        -- if UCO is currently onboarding (i.e. the onboarding date is in the past), this is the estimated dbus for the full current month. 
        else round(estimated_monthly_dollar_dbus * try_divide(datediff(getdate(), target_onboarding_date_15), total_ramping_days)) 
      end as current_dbu_baseline 

    --calculate the ramping dbus assuming a linear ramp between the tonboarding date and the go-live: from 0 $dbus to the expected monthly $dbus that will be reached on go-live.
    , case       
        when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
        when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus --After the go-live the $dbus remain flat 
        else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) --Between onboarding and go-live the dbus ramp-up linearly
      end as ramping_dbus

    --remove the realised dbus from the ramp, when the use case is ramping up during the onboarding phase, past months' revenue has already been realised.
    , case 
      when d.last_day_of_month < getdate() then 0 --Past month: if a use case started onboarding in the past, and the month is closed then we are removing the consumption from the pipeline to avoid double counting, because we assume it has already been realised (actual dbus).
      when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
      when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus - current_dbu_baseline --After the go-live
      else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) - current_dbu_baseline --Between onboarding and go-live 
    end as ramping_dbus_from_baseline 

     -- calculates the actual incremental value substracting last month's $dbus from this month's $dbus.
    , case 
        when getdate() > m then ramping_dbus - ramping_dbus_from_baseline
        else 0 --in the future
    end as dbus_generated

  from usecases_filtered as uco
  inner join dates as d --cross join with date table
),

quarterly_projection_by_use_case as (
  select              
    i.ae_email, i.deployable_account_name, i.usecase_id, i.fy, i.fq, i.q
    , sum(i.ramping_dbus) as quarterly_ramping_dbus    
    , sum(i.dbus_generated) as quarterly_dbus_generated
    , lag(max(i.ramping_dbus)) over (partition by i.ae_email, i.usecase_id order by i.fq asc) as last_day_of_prev_quarter_dbus
    from incremental_projections as i    
    group by all
),

monthly_projection as (
  select
    ip.ae_email, ip.deployable_account_name, ip.usecase_id, ip.fy, ip.fq, ip.q, ip.m, ip.cm, ip.ramping_dbus, ip.current_dbu_baseline, ip.is_onboarding
    , f.last_closed_q, f.days_left_in_quarter, f.is_quarter_closed, f.is_current_fiscal_quarter
    , qp.last_day_of_prev_quarter_dbus    
    , greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) as quarter_dbu_baseline

    -- Incremental quarterly $dbus. 
    -- If the quarter has started then we use the current baseline to identify addtional incremental dbus until the end of the quarter
    -- if the quarter has not started yet the baseline is the last day of the previous quarter.    
    ,case 
        when ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) < 0 then 0 -- All past months do not contribute to incremental dbus. 
        else ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline)
     end as quarterly_incremental_dbus

    , case when ip.implementation_status = 'Green' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_green 
    , case when ip.implementation_status = 'Yellow' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_yellow 
    , case when ip.implementation_status = 'Red' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_red
    , case when ip.implementation_status = 'Unknown' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_unknown
  
  from incremental_projections as ip
  inner join quarterly_projection_by_use_case as qp
  on ip.usecase_id = qp.usecase_id
  and ip.fq = qp.fq
  and ip.ae_email = qp.ae_email
  inner join financial_quarters as f
  on ip.fq = f.fq
),

monthly_projection_pivoted as (
  select ae_email, usecase_id
  , coalesce(`2025_02`, 0) as `2025_02`, coalesce(`2025_03`, 0) as `2025_03`, coalesce(`2025_04`, 0) as `2025_04`, coalesce(`2025_05`, 0) as `2025_05`, coalesce(`2025_06`, 0) as `2025_06`, coalesce(`2025_07`, 0) as `2025_07`
  , coalesce(`2025_08`, 0) as `2025_08`, coalesce(`2025_09`, 0) as `2025_09`, coalesce(`2025_10`, 0) as `2025_10`, coalesce(`2025_11`, 0) as `2025_11`, coalesce(`2025_12`, 0) as `2025_12`, coalesce(`2026_01`, 0) as `2026_01`
  , coalesce(`2026_02`, 0) as `2026_02`, coalesce(`2026_03`, 0) as `2026_03`, coalesce(`2026_04`, 0) as `2026_04`, coalesce(`2026_05`, 0) as `2026_05`, coalesce(`2026_06`, 0) as `2026_06`, coalesce(`2026_07`, 0) as `2026_07`
  , coalesce(`2026_08`, 0) as `2026_08`, coalesce(`2026_09`, 0) as `2026_09`, coalesce(`2026_10`, 0) as `2026_10`, coalesce(`2026_11`, 0) as `2026_11`, coalesce(`2026_12`, 0) as `2026_12`, coalesce(`2027_01`, 0) as `2027_01`
  from 
  (
    select ae_email, usecase_id, m, ramping_dbus from monthly_projection
  )
  pivot (sum(ramping_dbus) AS dbus
      for m in (
        '2025-02-01' as `2025_02`,
        '2025-03-01' as `2025_03`,
        '2025-04-01' as `2025_04`,
        '2025-05-01' as `2025_05`,
        '2025-06-01' as `2025_06`,
        '2025-07-01' as `2025_07`,
        '2025-08-01' as `2025_08`,
        '2025-09-01' as `2025_09`,
        '2025-10-01' as `2025_10`,
        '2025-11-01' as `2025_11`,
        '2025-12-01' as `2025_12`,
        '2026-01-01' as `2026_01`,
        '2026-02-01' as `2026_02`,
        '2026-03-01' as `2026_03`,
        '2026-04-01' as `2026_04`,
        '2026-05-01' as `2026_05`,
        '2026-06-01' as `2026_06`,
        '2026-07-01' as `2026_07`,
        '2026-08-01' as `2026_08`,
        '2026-09-01' as `2026_09`,
        '2026-10-01' as `2026_10`,
        '2026-11-01' as `2026_11`,
        '2026-12-01' as `2026_12`,
        '2027-01-01' as `2027_01`
      )
  )
),

-- Quarterly pipeline projection at deployable account level
quarterly_projection as (
  select p.ae_email, p.deployable_account_name, p.fy, p.fq, p.q, p.last_closed_q, p.days_left_in_quarter, p.is_quarter_closed, p.is_current_fiscal_quarter
    , sum(p.quarterly_incremental_dbus) as quarterly_incremental_dbus
    , sum(p.dbus_in_pipeline_green) as dbus_in_pipeline_green 
    , sum(p.dbus_in_pipeline_yellow) as dbus_in_pipeline_yellow 
    , sum(p.dbus_in_pipeline_red) as dbus_in_pipeline_red
    , sum(p.dbus_in_pipeline_unknown) as dbus_in_pipeline_unknown
    , sum(last_day_of_prev_quarter_dbus) as last_day_of_prev_quarter_dbus

    from monthly_projection as p
    group by all
),

quartely_projection_pivoted as (
  select ae_email, usecase_id
  , coalesce(FY26_Q1, 0) FY26_Q1
  , coalesce(FY26_Q2, 0) FY26_Q2
  , coalesce(FY26_Q3, 0) FY26_Q3
  , coalesce(FY26_Q4, 0) FY26_Q4
  , coalesce(FY27_Q1, 0) FY27_Q1
  , coalesce(FY27_Q2, 0) FY27_Q2
  , coalesce(FY27_Q3, 0) FY27_Q3
  , coalesce(FY27_Q4, 0) FY27_Q4
  from 
  (
    select ae_email, usecase_id, fq, sum(quarterly_incremental_dbus) as quarterly_incremental_dbus 
    from monthly_projection
    group by all
  )
  pivot (sum(quarterly_incremental_dbus) AS dbus
      for fq in (
        'FY26-Q1' as FY26_Q1,
        'FY26-Q2' as FY26_Q2,
        'FY26-Q3' as FY26_Q3,
        'FY26-Q4' as FY26_Q4,
        'FY27-Q1' as FY27_Q1,
        'FY27-Q2' as FY27_Q2,
        'FY27-Q3' as FY27_Q3,
        'FY27-Q4' as FY27_Q4
      )
  )
),

uco_view as (
  select b.ae_email, c.sales_subregion_level_1, c.sales_subregion_level_2, c.sales_subregion_level_3, c.account_name, c.account_executive, c.solution_architect, c.dsa, c.arr_band, c.usecase_id, c.usecase_name 
    , c.target_onboarding_date, b.target_onboarding_date_fq, c.target_live_date, b.target_live_date_fq, c.target_cloud, c.is_migration_usecase, c.is_incremental
    , c.stage, c.stage_number, c.stage_name_ui, c.days_in_stage, c.days_in_validating, c.days_in_scoping, c.days_in_evaluating, c.days_in_confirming, c.days_in_onboarding
    , c.usecase_description, c.demand_plan_next_steps, c.implementation_notes
    , c.implementation_partner_name, c.has_ps_project, c.use_case_product_enriched
    , b.estimated_monthly_dollar_dbus, c.estimated_monthly_dollar_dbus_weighted, b.total_ramping_days --, b.current_dbu_baseline, b.current_ramping_days
    , c.estimated_quarterly_dollar_dbus, c.estimated_quarterly_dollar_dbus_weighted
    , b.days_in_stage_bucket, b.usecase_url, b.implementation_status, b.num_of_blockers
    , get(FILTER(c.usecase_documents, doc -> doc.document_type = 'Eval Doc'), 0).document_link as eval_doc_link
    , get(FILTER(c.usecase_documents, doc -> doc.document_type = 'Onboarding Doc'), 0).document_link as onboarding_doc_link
    , qp.FY26_Q1, qp.FY26_Q2, qp.FY26_Q3, qp.FY26_Q4, qp.FY27_Q1, qp.FY27_Q2, qp.FY27_Q3, qp.FY27_Q4
    , mp.`2025_02`, mp.`2025_03`, mp.`2025_04`, mp.`2025_05`, mp.`2025_06`, mp.`2025_07`, mp.`2025_08`, mp.`2025_09`, mp.`2025_10`, mp.`2025_11`, mp.`2025_12`
    , mp.`2026_01`, mp.`2026_02`, mp.`2026_03`, mp.`2026_04`, mp.`2026_05`, mp.`2026_06`, mp.`2026_07`, mp.`2026_08`, mp.`2026_09`, mp.`2026_10`, mp.`2026_11`, mp.`2026_12`, mp.`2027_01`
    , b.days_to_go_live
    , b.days_to_onboarding
    , uco_info.category
    , uco_info.msg

    from gtm_silver.use_case_detail as c
    inner join usecases_filtered as b
    on c.usecase_id = b.usecase_id
    inner join quartely_projection_pivoted as qp
    on qp.usecase_id = c.usecase_id and qp.ae_email = b.ae_email
    inner join monthly_projection_pivoted as mp
    on mp.usecase_id = c.usecase_id and mp.ae_email = b.ae_email
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.usecase_id ORDER BY b.ae_email) = 1
), 

--this is used to build a Gantt Chart
uco_view_unpivoted as (
  select b.ae_email, mp.m, mp.fq, mp.cm, mp.is_onboarding, c.account_name, c.account_executive, c.solution_architect, c.arr_band, c.usecase_name 
    , c.target_onboarding_date, b.target_onboarding_date_fq, c.target_live_date, b.target_live_date_fq
    , c.stage, c.days_in_stage, b.days_in_stage_bucket, b.implementation_status, b.num_of_blockers
    , c.estimated_monthly_dollar_dbus
    from gtm_silver.use_case_detail as c
    inner join usecases_filtered as b
    on c.usecase_id = b.usecase_id
    inner join monthly_projection as mp
    on mp.usecase_id = c.usecase_id and mp.ae_email = b.ae_email
    where mp.is_onboarding > 0
), 

-- Quarterly summary at deployable account level - all metrics natively at this grain
quarterly_summary as (
  SELECT 
  f.ae_email, ae.sales_level, ae.user_name, f.deployable_account_name, d.fy, d.fq, d.q, d.days_left_in_quarter, d.is_quarter_closed, d.is_current_fiscal_quarter
  , f.fin_target, f.submitted_my_call 
  , f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
  , f.dbu_actuals_coalesced as dbu_actuals, f.dbu_actuals_or_forecast, f.dbu_actuals_current_quarter
  , f.dbu_dollars_t7d_adj, f.dbu_dollars_t7d_prev_adj, f.t7d_proj_left_in_quarter, f.target_t7d
  , f.dbu_dollars_t28d_adj, f.dbu_dollars_t28d_prev_adj, f.t28d_proj_left_in_quarter

  , coalesce(lag(f.dbu_actuals_or_forecast) over (partition by f.ae_email, f.deployable_account_name order by d.fq asc), 0) as dbu_actuals_or_forecast_prev_quarter
  , try_divide(f.fin_target - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter) as qoq_target_growth

  --incremental pipeline (natively at deployable account level via use case linkage)
  , coalesce(p.quarterly_incremental_dbus, 0) as quarterly_incremental_dbus
  , coalesce(p.dbus_in_pipeline_green, 0) as dbus_in_pipeline_green
  , coalesce(p.dbus_in_pipeline_yellow, 0) as dbus_in_pipeline_yellow
  , coalesce(p.dbus_in_pipeline_red, 0) as dbus_in_pipeline_red
  , coalesce(p.dbus_in_pipeline_unknown, 0) as dbus_in_pipeline_unknown
  , coalesce(p.dbus_in_pipeline_green, 0) * :green_confidence_pct as dbus_in_pipeline_green_in_plan
  , coalesce(p.dbus_in_pipeline_yellow, 0) * :yellow_confidence_pct as dbus_in_pipeline_yellow_in_plan
  , coalesce(p.dbus_in_pipeline_red, 0) * :red_confidence_pct as dbus_in_pipeline_red_in_plan
  , coalesce(p.dbus_in_pipeline_unknown, 0) * :unknown_confidence_pct as dbus_in_pipeline_unknown_in_plan

  --set the baseline: 
  --for the current quarter, use the T7D because more precise. 
  --for future quarters use 
  , case when d.is_quarter_closed then 0 when d.is_current_fiscal_quarter then f.t7d_proj_left_in_quarter else dbu_actuals_or_forecast_prev_quarter  end as baseline
  , case when d.is_quarter_closed then "Baseline" when d.is_current_fiscal_quarter then "Baseline (T7D Projection + OG)" else "Baseline (previous quarter's forecast) + OG" end as baseline_label -- used as a label in the dashboard

  , case when d.is_quarter_closed then 0 else f.dbu_actuals_coalesced + f.t7d_proj_left_in_quarter end as t7d_flat_projection 
  , case when d.is_quarter_closed then 0 else f.dbu_actuals_coalesced + f.t28d_proj_left_in_quarter end as t28d_flat_projection

  --best case
  , baseline * :best_case_qoq_organic_growth as best_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan + dbus_in_pipeline_yellow_in_plan + dbus_in_pipeline_red_in_plan + dbus_in_pipeline_unknown_in_plan as best_case_pipe_left_in_quarter
  , f.dbu_actuals_coalesced + best_case_proj_with_og_left_in_quarter + best_case_pipe_left_in_quarter + :best_case_adjustments as best_case_projection

  --worst case
  , baseline * :worst_case_qoq_organic_growth as worst_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan as worst_case_pipe_left_in_quarter
  , f.dbu_actuals_coalesced + worst_case_proj_with_og_left_in_quarter + worst_case_pipe_left_in_quarter + :worst_case_adjustments as worst_case_projection
  
  --labels
  , format_number(try_divide(f.submitted_my_call, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(f.submitted_my_call - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(f.submitted_my_call - f.fin_target, '$,###.#') as my_call_text
  , format_number(try_divide(f.submitted_direct_field_consumption_forecast, f.fin_target), '#.#%') ||
    " | " || format_number(try_divide(f.submitted_direct_field_consumption_forecast - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') ||
    " | " || format_number(f.submitted_direct_field_consumption_forecast - f.fin_target, '$,###.#') as directs_fct_text
  , format_number(try_divide(best_case_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(best_case_projection - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(best_case_projection - f.fin_target, '$,###.#') as best_case_text
  , format_number(try_divide(worst_case_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(worst_case_projection - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(worst_case_projection - f.fin_target, '$,###.#') as worst_case_text
  , format_number(try_divide(f.submitted_weighted_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(f.submitted_weighted_projection - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(f.submitted_weighted_projection - f.fin_target, '$,###.#') as weighted_proj_text
  , format_number(try_divide(f.current_ds_forecast, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(f.current_ds_forecast - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(f.current_ds_forecast - f.fin_target, '$,###.#') as ds_forecast_text
  , format_number(try_divide(t7d_flat_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(t7d_flat_projection - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(t7d_flat_projection - f.fin_target, '$,###.#') as t7d_proj_text
  , format_number(try_divide(t28d_flat_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(t28d_flat_projection - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(t28d_flat_projection - f.fin_target, '$,###.#') as t28d_proj_text

  --forecast gaps
  , f.submitted_my_call - f.fin_target as gap_to_target 
  , f.submitted_my_call - best_case_projection as gap_to_best_case
  , f.submitted_my_call - worst_case_projection as gap_to_worst_case
  , f.submitted_my_call - t7d_flat_projection as gap_to_t7d_flat_projection
  , f.submitted_my_call - t28d_flat_projection as gap_to_t28d_flat_projection
  , f.submitted_my_call - f.submitted_weighted_projection as gap_to_weighted_projection
  , f.submitted_my_call - f.current_ds_forecast as gap_to_ds_forecast
  , f.submitted_my_call - f.submitted_direct_field_consumption_forecast as gap_to_directs_forecast

  from target_forecast_actuals as f
  inner join financial_quarters d on d.fiscal_quarter_start_date = f.fiscal_quarter_start_date
  left outer join quarterly_projection as p 
  on p.fy = f.fy and p.q = f.q and p.ae_email = f.ae_email and p.deployable_account_name = f.deployable_account_name
  inner join ae_list ae on ae.ae_email = f.ae_email
),

-- Aggregated AE-level rollup with deployable_account_name = 'All'
-- Uses targets_individual_ae for fin_target and forecast_individual_ae for submitted_my_call/directs
quarterly_summary_combined as (
  SELECT * FROM quarterly_summary
  UNION ALL
  SELECT
    qs.ae_email, qs.sales_level, qs.user_name, 'All' as deployable_account_name, qs.fy, qs.fq, qs.q, qs.days_left_in_quarter, qs.is_quarter_closed, qs.is_current_fiscal_quarter
    , COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)) as fin_target
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) as submitted_my_call
    , COALESCE(MAX(fia.submitted_direct_field_consumption_forecast), SUM(qs.submitted_direct_field_consumption_forecast)) as submitted_direct_field_consumption_forecast
    , sum(qs.current_ds_forecast) as current_ds_forecast
    , sum(qs.submitted_weighted_projection) as submitted_weighted_projection
    , sum(qs.dbu_actuals) as dbu_actuals
    , sum(qs.dbu_actuals_or_forecast) as dbu_actuals_or_forecast
    , sum(qs.dbu_actuals_current_quarter) as dbu_actuals_current_quarter
    , sum(qs.dbu_dollars_t7d_adj) as dbu_dollars_t7d_adj
    , sum(qs.dbu_dollars_t7d_prev_adj) as dbu_dollars_t7d_prev_adj
    , sum(qs.t7d_proj_left_in_quarter) as t7d_proj_left_in_quarter
    , sum(qs.target_t7d) as target_t7d
    , sum(qs.dbu_dollars_t28d_adj) as dbu_dollars_t28d_adj
    , sum(qs.dbu_dollars_t28d_prev_adj) as dbu_dollars_t28d_prev_adj
    , sum(qs.t28d_proj_left_in_quarter) as t28d_proj_left_in_quarter
    , sum(qs.dbu_actuals_or_forecast_prev_quarter) as dbu_actuals_or_forecast_prev_quarter
    , try_divide(COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)) as qoq_target_growth
    , sum(qs.quarterly_incremental_dbus) as quarterly_incremental_dbus
    , sum(qs.dbus_in_pipeline_green) as dbus_in_pipeline_green
    , sum(qs.dbus_in_pipeline_yellow) as dbus_in_pipeline_yellow
    , sum(qs.dbus_in_pipeline_red) as dbus_in_pipeline_red
    , sum(qs.dbus_in_pipeline_unknown) as dbus_in_pipeline_unknown
    , sum(qs.dbus_in_pipeline_green_in_plan) as dbus_in_pipeline_green_in_plan
    , sum(qs.dbus_in_pipeline_yellow_in_plan) as dbus_in_pipeline_yellow_in_plan
    , sum(qs.dbus_in_pipeline_red_in_plan) as dbus_in_pipeline_red_in_plan
    , sum(qs.dbus_in_pipeline_unknown_in_plan) as dbus_in_pipeline_unknown_in_plan
    , sum(qs.baseline) as baseline
    , max(qs.baseline_label) as baseline_label
    , sum(qs.t7d_flat_projection) as t7d_flat_projection
    , sum(qs.t28d_flat_projection) as t28d_flat_projection
    , sum(qs.best_case_proj_with_og_left_in_quarter) as best_case_proj_with_og_left_in_quarter
    , sum(qs.best_case_pipe_left_in_quarter) as best_case_pipe_left_in_quarter
    , sum(qs.best_case_projection) as best_case_projection
    , sum(qs.worst_case_proj_with_og_left_in_quarter) as worst_case_proj_with_og_left_in_quarter
    , sum(qs.worst_case_pipe_left_in_quarter) as worst_case_pipe_left_in_quarter
    , sum(qs.worst_case_projection) as worst_case_projection
    -- text fields recomputed from aggregated values using individual target and individual forecasts
    , format_number(try_divide(COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as my_call_text
    , format_number(try_divide(COALESCE(MAX(fia.submitted_direct_field_consumption_forecast), SUM(qs.submitted_direct_field_consumption_forecast)), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') ||
      " | " || format_number(try_divide(COALESCE(MAX(fia.submitted_direct_field_consumption_forecast), SUM(qs.submitted_direct_field_consumption_forecast)) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') ||
      " | " || format_number(COALESCE(MAX(fia.submitted_direct_field_consumption_forecast), SUM(qs.submitted_direct_field_consumption_forecast)) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as directs_fct_text
    , format_number(try_divide(sum(qs.best_case_projection), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.best_case_projection) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.best_case_projection) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as best_case_text
    , format_number(try_divide(sum(qs.worst_case_projection), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.worst_case_projection) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.worst_case_projection) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as worst_case_text
    , format_number(try_divide(sum(qs.submitted_weighted_projection), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.submitted_weighted_projection) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.submitted_weighted_projection) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as weighted_proj_text
    , format_number(try_divide(sum(qs.current_ds_forecast), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.current_ds_forecast) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.current_ds_forecast) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as ds_forecast_text
    , format_number(try_divide(sum(qs.t7d_flat_projection), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.t7d_flat_projection) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.t7d_flat_projection) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as t7d_proj_text
    , format_number(try_divide(sum(qs.t28d_flat_projection), COALESCE(MAX(tia.fin_target), SUM(qs.fin_target))), '#.#%') || 
      " | " || format_number(try_divide(sum(qs.t28d_flat_projection) - sum(qs.dbu_actuals_or_forecast_prev_quarter), sum(qs.dbu_actuals_or_forecast_prev_quarter)), '#.#%') || 
      " | " || format_number(sum(qs.t28d_flat_projection) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)), '$,###.#') as t28d_proj_text
    -- gaps recomputed using individual my_call and individual target
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - COALESCE(MAX(tia.fin_target), SUM(qs.fin_target)) as gap_to_target
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.best_case_projection) as gap_to_best_case
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.worst_case_projection) as gap_to_worst_case
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.t7d_flat_projection) as gap_to_t7d_flat_projection
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.t28d_flat_projection) as gap_to_t28d_flat_projection
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.submitted_weighted_projection) as gap_to_weighted_projection
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - sum(qs.current_ds_forecast) as gap_to_ds_forecast
    , COALESCE(MAX(fia.submitted_my_call), SUM(qs.submitted_my_call)) - COALESCE(MAX(fia.submitted_direct_field_consumption_forecast), SUM(qs.submitted_direct_field_consumption_forecast)) as gap_to_directs_forecast
  FROM quarterly_summary qs
  LEFT JOIN targets_individual_ae tia ON tia.ae_email = qs.ae_email AND tia.fy = qs.fy AND tia.q = qs.q
  LEFT JOIN forecast_individual_ae fia ON fia.ae_email = qs.ae_email AND fia.fy = qs.fy AND fia.q = qs.q
  GROUP BY qs.ae_email, qs.sales_level, qs.user_name, qs.fy, qs.fq, qs.q, qs.days_left_in_quarter, qs.is_quarter_closed, qs.is_current_fiscal_quarter
)


select * from uco_view

/* Forecast waterfall for current user only */
/*select * from quarterly_summary_combined 
where ae_email = :ae_email
AND deployable_account_name = :account_name
*/

--Forecast waterfall for current user and all its AEs
/*
select fy, fq, ae_email, user_name,  sales_level, submitted_my_call, fin_target, submitted_my_call, submitted_direct_field_consumption_forecast, worst_case_projection, best_case_projection gaps, value
from quarterly_summary_combined
unpivot (
  value for gaps in (
    gap_to_target as `Gap to Target`,
    gap_to_ds_forecast as `Gap to DS`,
    gap_to_worst_case as `Gap to Worst Case`,
    gap_to_best_case as `Gap to Best Case`,
    gap_to_t7d_flat_projection as `Gap to T7D Proj`,
    gap_to_weighted_projection as `Gap to WP`
  )
)*/


--select * from uco_view_unpivoted